In [1]:
import numpy as np

def volume_from_grid_axes(grid, axes):
    """
    Convert our (nx, ny, nz) grid + 1D axes into napari-friendly stuff.
    napari expects (Z, Y, X) array plus an affine or (scale, translate) for world coords.

    Returns
    -------
    volume : (nz, ny, nx) ndarray
    scale  : (dz, dy, dx) voxel size along z, y, x
    translate : (z0, y0, x0) world coord of voxel (0,0,0)
    is_uniform : bool (whether axes are strictly uniform; if False we used average spacing)
    """
    xax, yax, zax = [np.asarray(a) for a in axes]   # lengths nx, ny, nz
    nx, ny, nz = len(xax), len(yax), len(zax)
    assert grid.shape == (nx, ny, nz)

    # napari expects zyx ordering
    volume = grid.transpose(2, 1, 0).copy()  # -> (nz, ny, nx)

    # compute spacing; if non-uniform, use average (napari only supports linear transforms)
    def _spacing(a):
        d = np.diff(a)
        return float(d.mean()), bool(np.allclose(d, d[0], rtol=1e-5, atol=1e-8))
    dx, x_uni = _spacing(xax)
    dy, y_uni = _spacing(yax)
    dz, z_uni = _spacing(zax)
    is_uniform = x_uni and y_uni and z_uni

    # scale is (dz, dy, dx) for (Z,Y,X)
    scale = (dz, dy, dx)
    translate = (float(zax[0]), float(yax[0]), float(xax[0]))
    return volume, scale, translate, is_uniform


def log1p_clip(a):
    """Nice-looking intensity for volume rendering."""
    a = np.asarray(a)
    a = np.maximum(a, 0.0)
    return np.log1p(a)

In [ ]:
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
spec_file = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23"
tiff_dir  = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/data_6oct23_tiff"
scan_list = (17, 18, 19)     # any list/tuple of scan numbers
out_vtr   = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/rsm_hkl.vtr"    # output file

# Build with 4-circle (ZXZ: φ(Z) → χ(X) → ω(Z))
builder = RSMBuilder(
    spec_file, tiff_dir,
    selected_scans=scan_list,
    ub_includes_2pi=True,        # set False if your UB is "no-2π"
    center_is_one_based=False,   # True if SPEC xcenter/ycenter are 1-based
    fourc_mode="ZXZ",            # or "ZYX" if your instrument uses Z-Y-X
    motor_map={"omega":"th", "chi":"chi", "phi":"phi"},  # map to your SPEC columns
)

# Compute per-pixel Q & HKL
Q_samp, hkl, intensity = builder.compute_full()

# Regrid with xrayutilities gridder (mean or sum)
grid, (xax, yax, zax) = builder.regrid_xu(
    space="hkl",                 # "hkl" or "q"
    grid_shape=(200, 200, 200),  # adjust to taste / memory
    ranges=None,                 # or ((xmin,xmax),(ymin,ymax),(zmin,zmax)) to lock axes
    fuzzy=False,                 # True → FuzzyGridder3D; add width=... for footprint
    normalize="mean",            # "mean" or "sum"
    stream=True                  # frame-by-frame accumulation (RAM friendly)
)
grid.shape, len(xax), len(yax), len(zax)


((200, 200, 200), 200, 200, 200)

In [5]:
# Assume you've already done:
# grid, (xax, yax, zax) = builder.regrid_xu(space="hkl", grid_shape=(256,256,256), normalize="mean", stream=True)

import napari

volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))
v = napari.Viewer(ndisplay=3)
v.add_image(
    log1p_clip(volume),               # 3D image (Z,Y,X)
    name="RSM (log1p)", 
    colormap="viridis",
    rendering="attenuated_mip",       # alternatives: 'mip', 'translucent'
    opacity=1.0,
    scale=scale,                      # voxel spacing
    translate=translate               # world origin
)

# Add an outline layer to show the data bounds
import numpy as np
shape = volume.shape  # (nz, ny, nx)
z0, y0, x0 = translate
dz, dy, dx = scale
# Compute the 8 corners of the volume in world coordinates
corners = np.array([[z, y, x] for z in [0, shape[0]-1] for y in [0, shape[1]-1] for x in [0, shape[2]-1]])
corners_world = corners * np.array([dz, dy, dx]) + np.array([z0, y0, x0])
v.add_points(corners_world, size=2, face_color='red', name='Outline corners')

# Optionally, add lines to connect the corners (outline box)
box_lines = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7]
])
# Set edge_width to 1 for a thinner outline
v.add_shapes([corners_world[line] for line in box_lines], shape_type='line', edge_color='yellow', edge_width=1, name='Outline box')

# Add coordinate axes as vectors (fix napari API: use shape (N,2,3))
axes_length = 0.1 * max(dz*shape[0], dy*shape[1], dx*shape[2])
origin = np.array([z0, y0, x0])
axes_vecs = np.array([[axes_length,0,0],[0,axes_length,0],[0,0,axes_length]])
vectors = np.stack([np.tile(origin, (3,1)), origin + axes_vecs], axis=1)  # shape (3,2,3)
v.add_vectors(vectors, edge_color=['red','green','blue'], name='Axes')

# Optional: also add the linear (non-log) version
# v.add_image(volume, name="RSM (linear)", colormap="magma", rendering="mip", scale=scale, translate=translate)

print("Uniform spacing:", is_uniform)  # if False, we used average spacing (linear approx)

Uniform spacing: True
 True


In [3]:
import numpy as np
import napari

# ---- helpers ---------------------------------------------------------------
def volume_from_grid_axes(grid, axes):
    """
    Convert (nx,ny,nz) + 1D axes -> (nz,ny,nx) volume for napari
    with linear world transform (scale, translate).
    Uses average spacing if axes are not perfectly uniform.
    """
    xax, yax, zax = [np.asarray(a) for a in axes]  # lengths nx, ny, nz
    nx, ny, nz = len(xax), len(yax), len(zax)
    assert grid.shape == (nx, ny, nz)

    # napari image expects (Z, Y, X)
    volume = grid.transpose(2, 1, 0).copy()

    def _avg_spacing(a):
        return float(np.diff(a).mean()) if len(a) > 1 else 1.0

    dx = _avg_spacing(xax)
    dy = _avg_spacing(yax)
    dz = _avg_spacing(zax)

    # world origin at the first coordinate of each axis
    translate = (float(zax[0]), float(yax[0]), float(xax[0]))   # (Z,Y,X)
    scale     = (dz, dy, dx)                                    # voxel size (Z,Y,X)
    is_uniform = (
        np.allclose(np.diff(xax), dx, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(yax), dy, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(zax), dz, rtol=1e-5, atol=1e-8)
    )
    return volume, scale, translate, is_uniform

def log1p_clip(a):
    a = np.asarray(a)
    return np.log1p(np.maximum(a, 0.0))

# Compute napari data + transform
volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))

# ---- viewer + volume layer --------------------------------------------------
v = napari.Viewer(ndisplay=3, title="RSM viewer")
# nice starting contrast on log data
vol_log = log1p_clip(volume)
lo, hi = np.percentile(vol_log, [1, 99.8])

layer = v.add_image(
    vol_log,                        # (Z,Y,X)
    name="RSM (log1p)",
    colormap="viridis",
    rendering="attenuated_mip",     # 'mip' or 'translucent' also good
    blending="translucent",
    opacity=1.0,
    scale=scale,                    # voxel spacing (Z,Y,X)
    translate=translate,            # world origin (Z,Y,X)
    contrast_limits=(float(lo), float(hi)),
)

# axes UI / scale bar / labels
v.axes.visible = True
v.axes.colored = True
v.axes.arrows = True
v.scale_bar.visible = True
# Set units: "" for HKL (dimensionless), "Å⁻¹" for Q-space
v.scale_bar.unit = ""               # change to "Å⁻¹" if visualizing Q
# Label displayed dims (Z,Y,X). For HKL use ("L","K","H"); for Q use ("Qz","Qy","Qx")
v.dims.axis_labels = ("L", "K", "H")

# ---- world-space outline box -----------------------------------------------
# Use actual world bounds from axes (works for non-uniform axes too)
# --- replace your "outline/corners/axes" block with this ---

# World bounds from axes (handles non-uniform axes)
zmin, zmax = float(zax[0]), float(zax[-1])
ymin, ymax = float(yax[0]), float(yax[-1])
xmin, xmax = float(xax[0]), float(xax[-1])

# 8 corners in world coords (Z,Y,X for napari)
corners_world = np.array([
    [zmin, ymin, xmin],
    [zmin, ymin, xmax],
    [zmin, ymax, xmin],
    [zmin, ymax, xmax],
    [zmax, ymin, xmin],
    [zmax, ymin, xmax],
    [zmax, ymax, xmin],
    [zmax, ymax, xmax],
], dtype=float)

# ---- THIN CORNERS (Points) ---------------------------------------------------
# Make the markers << 1 voxel in each axis so they stay tiny.
voxel = np.array(scale, dtype=float)            # (dz, dy, dx)
corner_size_xyz = voxel * 0.25                  # quarter-voxel ellipsoids
corner_sizes = np.repeat(corner_size_xyz[None, :], 8, axis=0)  # shape (8, 3)

v.add_points(
    corners_world,
    name="Outline corners",
    size=corner_sizes,          # anisotropic per-point sizes (Z,Y,X)
    edge_width=0,               # no outline
    face_color="red",
    opacity=0.9,
    blending="additive",
)

# ---- ULTRA-THIN OUTLINE (Shapes: lines) -------------------------------------
box_edges = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7],
], dtype=int)
edge_segments = [corners_world[e] for e in box_edges]

v.add_shapes(
    edge_segments,
    shape_type="line",
    edge_color="yellow",
    edge_width=0.5,                 # sub-pixel hairline
    edge_width_is_relative=False,   # width in screen pixels
    opacity=0.9,
    blending="additive",
    name="Outline box",
)

# ---- SLIM AXES (Vectors) ----------------------------------------------------
# axis length ~10% of largest world extent
Lx, Ly, Lz = xmax - xmin, ymax - ymin, zmax - zmin
axes_len = 0.10 * max(Lx, Ly, Lz)
origin = np.array([zmin, ymin, xmin], dtype=float)

vectors = np.stack([
    np.vstack([origin, origin + np.array([axes_len, 0, 0])]),  # +Z (cyan)
    np.vstack([origin, origin + np.array([0, axes_len, 0])]),  # +Y (lime)
    np.vstack([origin, origin + np.array([0, 0, axes_len])]),  # +X (magenta)
], axis=0)  # (3, 2, 3)

v.add_vectors(
    vectors,
    name="World axes",
    edge_color=["cyan", "lime", "magenta"],
    width=0.75,                 # pixel width; <1 is OK for fine lines
    blending="translucent_no_depth",
)


print("Uniform spacing:", is_uniform)  # if False, we used average spacing in the linear transform
v

TypeError: add_points() got an unexpected keyword argument 'edge_width'

In [ ]:
# --- replace your "outline/corners/axes" block with this ---

# World bounds from axes (handles non-uniform axes)
zmin, zmax = float(zax[0]), float(zax[-1])
ymin, ymax = float(yax[0]), float(yax[-1])
xmin, xmax = float(xax[0]), float(xax[-1])

# 8 corners in world coords (Z,Y,X for napari)
corners_world = np.array([
    [zmin, ymin, xmin],
    [zmin, ymin, xmax],
    [zmin, ymax, xmin],
    [zmin, ymax, xmax],
    [zmax, ymin, xmin],
    [zmax, ymin, xmax],
    [zmax, ymax, xmin],
    [zmax, ymax, xmax],
], dtype=float)

# ---- THIN CORNERS (Points) ---------------------------------------------------
# Make the markers << 1 voxel in each axis so they stay tiny.
voxel = np.array(scale, dtype=float)            # (dz, dy, dx)
corner_size_xyz = voxel * 0.25                  # quarter-voxel ellipsoids
corner_sizes = np.repeat(corner_size_xyz[None, :], 8, axis=0)  # shape (8, 3)

v.add_points(
    corners_world,
    name="Outline corners",
    size=corner_sizes,          # anisotropic per-point sizes (Z,Y,X)
    edge_width=0.1,               # no outline
    face_color="red",
    opacity=0.9,
    blending="additive",
)

# ---- ULTRA-THIN OUTLINE (Shapes: lines) -------------------------------------
box_edges = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7],
], dtype=int)
edge_segments = [corners_world[e] for e in box_edges]

v.add_shapes(
    edge_segments,
    shape_type="line",
    edge_color="yellow",
    edge_width=0.1,                 # sub-pixel hairline
    edge_width_is_relative=False,   # width in screen pixels
    opacity=0.9,
    blending="additive",
    name="Outline box",
)

# ---- SLIM AXES (Vectors) ----------------------------------------------------
# axis length ~10% of largest world extent
Lx, Ly, Lz = xmax - xmin, ymax - ymin, zmax - zmin
axes_len = 0.10 * max(Lx, Ly, Lz)
origin = np.array([zmin, ymin, xmin], dtype=float)

vectors = np.stack([
    np.vstack([origin, origin + np.array([axes_len, 0, 0])]),  # +Z (cyan)
    np.vstack([origin, origin + np.array([0, axes_len, 0])]),  # +Y (lime)
    np.vstack([origin, origin + np.array([0, 0, axes_len])]),  # +X (magenta)
], axis=0)  # (3, 2, 3)

v.add_vectors(
    vectors,
    name="World axes",
    edge_color=["cyan", "lime", "magenta"],
    width=0.75,                 # pixel width; <1 is OK for fine lines
    blending="translucent_no_depth",
)

TypeError: add_points() got an unexpected keyword argument 'edge_width'

In [4]:
# --- Minimal, crisp napari visualization for 3D RSM (HKL or Q) ----------------
import numpy as np
import napari

# ---------- helpers ----------
def volume_from_grid_axes(grid, axes):
    """
    (nx,ny,nz) + 1D axes -> (nz,ny,nx) volume with (scale, translate) for napari.
    Uses average spacing if axes are not perfectly uniform (napari supports linear transforms).
    """
    xax, yax, zax = [np.asarray(a) for a in axes]
    nx, ny, nz = len(xax), len(yax), len(zax)
    if grid.shape != (nx, ny, nz):
        raise ValueError(f"grid shape {grid.shape} != ({nx},{ny},{nz}) from axes")

    volume = grid.transpose(2, 1, 0).copy()  # (Z,Y,X) for napari

    def _avg_step(a):
        return float(np.diff(a).mean()) if len(a) > 1 else 1.0

    dx = _avg_step(xax); dy = _avg_step(yax); dz = _avg_step(zax)
    translate = (float(zax[0]), float(yax[0]), float(xax[0]))  # (Z,Y,X)
    scale     = (dz, dy, dx)                                   # voxel size (Z,Y,X)
    is_uniform = (
        np.allclose(np.diff(xax), dx, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(yax), dy, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(zax), dz, rtol=1e-5, atol=1e-8)
    )
    return volume, scale, translate, is_uniform

def log1p_clip(a):
    a = np.asarray(a)
    return np.log1p(np.maximum(a, 0.0))

# ---------- volume + viewer ----------
# Assumes you already have: grid, (xax, yax, zax)
volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))

v = napari.Viewer(ndisplay=3, title="RSM viewer")

vol_log = log1p_clip(volume)
lo, hi = np.percentile(vol_log, [1, 99.8])
layer = v.add_image(
    vol_log,
    name="RSM (log1p)",
    colormap="viridis",
    rendering="attenuated_mip",   # or 'mip', 'translucent'
    blending="translucent",
    opacity=1.0,
    scale=scale,
    translate=translate,
    contrast_limits=(float(lo), float(hi)),
)

# UI niceties
v.axes.visible = True
v.axes.colored = True
v.axes.arrows = True
v.scale_bar.visible = True
# For HKL (dimensionless) leave empty ""; for Q use "Å⁻¹"
v.scale_bar.unit = ""      # set to "Å⁻¹" for Q-space
# Label dims (Z,Y,X). For HKL: ("L","K","H"); for Q: ("Qz","Qy","Qx")
v.dims.axis_labels = ("L", "K", "H")

# ---------- super-thin corners and outline (world coordinates) ----------
# True world bounds from axes (works for non-uniform axes)
zmin, zmax = float(zax[0]), float(zax[-1])
ymin, ymax = float(yax[0]), float(yax[-1])
xmin, xmax = float(xax[0]), float(xax[-1])

corners_world = np.array([
    [zmin, ymin, xmin],
    [zmin, ymin, xmax],
    [zmin, ymax, xmin],
    [zmin, ymax, xmax],
    [zmax, ymin, xmin],
    [zmax, ymin, xmax],
    [zmax, ymax, xmin],
    [zmax, ymax, xmax],
], dtype=float)

# Corners: make markers << 1 voxel so they stay tiny at any zoom
voxel = np.array(scale, dtype=float)             # (dz, dy, dx)
corner_size_xyz = voxel * 0.15                   # 0.15 voxel along each axis
corner_sizes = np.repeat(corner_size_xyz[None, :], 8, axis=0)  # (8,3) anisotropic

v.add_points(
    corners_world,
    name="Outline corners",
    size=corner_sizes,           # per-point anisotropic (Z,Y,X)
    edge_width=0,                # no outline
    face_color="red",
    opacity=0.9,
    blending="additive",
)

# Outline: 12 edges as ultra-thin lines (sub-pixel)
box_edges = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7],
], dtype=int)
edge_segments = [corners_world[e] for e in box_edges]

v.add_shapes(
    edge_segments,
    shape_type="line",
    edge_color="yellow",
    edge_width=0.5,                # hairline
    edge_width_is_relative=False,  # width in screen pixels
    opacity=0.9,
    blending="additive",
    name="Outline box",
)

# ---------- slim world-axis vectors ----------
Lx, Ly, Lz = xmax - xmin, ymax - ymin, zmax - zmin
axes_len = 0.10 * max(Lx, Ly, Lz)
origin = np.array([zmin, ymin, xmin], dtype=float)

# 3 vectors: +Z (cyan), +Y (lime), +X (magenta). Shape (N,2,3).
vectors = np.stack([
    np.vstack([origin, origin + np.array([axes_len, 0, 0])]),  # +Z
    np.vstack([origin, origin + np.array([0, axes_len, 0])]),  # +Y
    np.vstack([origin, origin + np.array([0, 0, axes_len])]),  # +X
], axis=0)

v.add_vectors(
    vectors,
    name="World axes",
    edge_color=["cyan", "lime", "magenta"],
    edge_width=0.75,                  # thin vector lines (pixels)
    blending="translucent_no_depth",
)

print("Uniform voxel spacing:", is_uniform)
v

TypeError: add_points() got an unexpected keyword argument 'edge_width'

In [4]:
import numpy as np
import napari

# ---------- helpers ----------
def volume_from_grid_axes(grid, axes):
    """
    (nx,ny,nz) + 1D axes -> (nz,ny,nx) volume plus (scale, translate) for napari.
    Uses average spacing for the linear world transform; exact coords come from axes below.
    """
    xax, yax, zax = [np.asarray(a) for a in axes]
    nx, ny, nz = len(xax), len(yax), len(zax)
    if grid.shape != (nx, ny, nz):
        raise ValueError(f"grid shape {grid.shape} != ({nx},{ny},{nz}) from axes")

    vol = grid.transpose(2, 1, 0).copy()  # (Z,Y,X)

    def _avg_step(a): return float(np.diff(a).mean()) if len(a) > 1 else 1.0
    dx, dy, dz = _avg_step(xax), _avg_step(yax), _avg_step(zax)

    translate = (float(zax[0]), float(yax[0]), float(xax[0]))  # (Z,Y,X)
    scale     = (dz, dy, dx)
    is_uniform = (
        np.allclose(np.diff(xax), dx, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(yax), dy, rtol=1e-5, atol=1e-8) and
        np.allclose(np.diff(zax), dz, rtol=1e-5, atol=1e-8)
    )
    return vol, scale, translate, is_uniform

def log1p_clip(a):
    a = np.asarray(a)
    return np.log1p(np.maximum(a, 0.0))

def index_to_axis_value(ax, idx):
    """
    Map a (possibly fractional) data index -> exact world coordinate using your axis array.
    Works for non-uniform axes.
    """
    n = len(ax)
    if n == 0:
        return np.nan
    if idx <= 0:
        return float(ax[0])
    if idx >= n - 1:
        return float(ax[-1])
    i0 = int(np.floor(idx))
    t  = float(idx - i0)
    return float((1.0 - t) * ax[i0] + t * ax[i0 + 1])

# ---------- build napari viewer ----------
# Assumes you already have: grid, (xax, yax, zax)
volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))

v = napari.Viewer(ndisplay=3, title="RSM viewer")

vol_log = log1p_clip(volume)
lo, hi = np.percentile(vol_log, [1, 99.8])  # quick sensible contrast
img_layer = v.add_image(
    vol_log,
    name="RSM (log1p)",
    colormap="viridis",
    rendering="attenuated_mip",
    blending="translucent",
    opacity=1.0,
    scale=scale,          # linear world transform (approx if non-uniform axes)
    translate=translate,
    contrast_limits=(float(lo), float(hi)),
)

# UI niceties
v.axes.visible = True
v.axes.colored = True
v.axes.arrows = True
v.scale_bar.visible = True
v.scale_bar.unit = ""              # set to "Å⁻¹" if you’re viewing Q-space
v.dims.axis_labels = ("L", "K", "H")  # for HKL; use ("Qz","Qy","Qx") for Q-space

# ---------- super-thin corners & outline (world coordinates) ----------
zmin, zmax = float(zax[0]), float(zax[-1])
ymin, ymax = float(yax[0]), float(yax[-1])
xmin, xmax = float(xax[0]), float(xax[-1])

corners_world = np.array([
    [zmin, ymin, xmin],
    [zmin, ymin, xmax],
    [zmin, ymax, xmin],
    [zmin, ymax, xmax],
    [zmax, ymin, xmin],
    [zmax, ymin, xmax],
    [zmax, ymax, xmin],
    [zmax, ymax, xmax],
], dtype=float)

# tiny corner markers: << 1 voxel (use isotropic size for napari stable)
voxel = np.array(scale, dtype=float)           # (dz, dy, dx)
corner_size = min(voxel) * 0.15                # 0.15 of smallest voxel size
corner_sizes = np.full(8, corner_size)         # (8,) isotropic marker size
v.add_points(
    corners_world,
    name="Outline corners",
    size=corner_sizes,      # isotropic (length 8)
    face_color="red",
    opacity=0.9,
    blending="additive",
)

# hairline outline
box_edges = np.array([
    [0,1],[0,2],[0,4],
    [1,3],[1,5],
    [2,3],[2,6],
    [3,7],
    [4,5],[4,6],
    [5,7],
    [6,7],
], dtype=int)
edge_segments = [corners_world[e] for e in box_edges]
v.add_shapes(
    edge_segments,
    shape_type="line",
    edge_color="yellow",
    edge_width=0.5,               # sub-pixel hairline
    opacity=0.9,
    blending="additive",
    name="Outline box",
)

# slim world-axis vectors
Lx, Ly, Lz = xmax - xmin, ymax - ymin, zmax - zmin
axes_len = 0.10 * max(Lx, Ly, Lz)
origin = np.array([zmin, ymin, xmin], dtype=float)
vectors = np.stack([
    np.vstack([origin, origin + np.array([axes_len, 0, 0])]),  # +Z
    np.vstack([origin, origin + np.array([0, axes_len, 0])]),  # +Y
    np.vstack([origin, origin + np.array([0, 0, axes_len])]),  # +X
], axis=0)
v.add_vectors(
    vectors,
    name="World axes",
    edge_color=["cyan", "lime", "magenta"],
    edge_width=0.75,
    blending="translucent_no_depth",
)

# ---------- live coordinate HUD (H, K, L + intensity under cursor) ----------
# Uses exact coordinate mapping from axis arrays (works for non-uniform axes).
# If napari version has viewer.text_overlay, we’ll use it; otherwise we print to console.
def on_mouse_move(viewer, event):
    pos_world = viewer.cursor.position
    if pos_world is None:
        return
    # data indices in (Z,Y,X) for this layer
    zi, yi, xi = img_layer.world_to_data(pos_world)
    # intensity sample (linear intensity from 'volume')
    I = np.nan
    zi_i, yi_i, xi_i = int(np.round(zi)), int(np.round(yi)), int(np.round(xi))
    if (0 <= zi_i < volume.shape[0]) and (0 <= yi_i < volume.shape[1]) and (0 <= xi_i < volume.shape[2]):
        I = float(volume[zi_i, yi_i, xi_i])

    # exact HKL (or Q) using axis arrays
    H = index_to_axis_value(xax, xi)
    K = index_to_axis_value(yax, yi)
    L = index_to_axis_value(zax, zi)
    text = f"H={H:.4f}   K={K:.4f}   L={L:.4f}    I={I:.3g}"

    if hasattr(viewer, "text_overlay") and viewer.text_overlay is not None:
        overlay = viewer.text_overlay
        overlay.visible = True
        overlay.position = 'top_left'
        overlay.color = 'white'
        overlay.font_size = 12
        overlay.text = text
    else:
        print(text, end="\r")

v.mouse_move_callbacks.append(on_mouse_move)

# Optional: press 'C' to toggle the HUD
@v.bind_key('C')
def _toggle_coords(viewer):
    if hasattr(viewer, "text_overlay") and viewer.text_overlay is not None:
        viewer.text_overlay.visible = not viewer.text_overlay.visible

print("Uniform voxel spacing (for the linear transform):", is_uniform)
v

Uniform voxel spacing (for the linear transform): True
 True


Viewer(camera=Camera(center=(np.float64(0.6432914733886737), np.float64(3.26029849052429), np.float64(2.264181137084959)), zoom=np.float64(20.250962096266157), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(0.0, 0.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=3, ndisplay=3, order=(0, 1, 2), axis_labels=('L', 'K', 'H'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(-20.705646514892578), stop=np.float64(11.639406204223633), step=np.float64(0.11051371588778855)), RangeTuple(start=np.float64(-10.871415138244629), stop=np.float64(11.956304550170898), step=np.float64(0.08739704582559403)), RangeTuple(start=np.float64(-23.47771453857422), stop=np.float64(16.26721954345703), step=np.float64(0.14073405433539768))), margin_left=(0